# Gatefall — LoRA training on Google Colab (kohya-ss/sd-scripts, free GPU)

Trains a per-character LoRA against whatever checkpoint you generated
your training images with, so future generations of that character
stay consistent across pose/crop/expression, instead of drifting the
way a bare fixed-seed prompt does once composition tokens change.

**Before running:** `Runtime` -> `Change runtime type` -> `T4 GPU` ->
`Save`. Run cells in order.

**Prerequisite — you need a training set first**, not just one locked
reference image. See `docs/art-direction.md`: generate 15-20 variants
of the locked character (different angles, expressions, crops) via
img2img at moderate denoise (~0.3-0.55, tune per how much the pose
needs to shift) off the locked reference image in the ComfyUI
notebook, so identity stays close across the set. Fresh txt2img
rerolls, even at the same seed, are not consistent enough once the
prompt's composition tokens change — that's the exact problem this
LoRA fixes going forward.

**Free-tier Colab is tight for SDXL LoRA training** — a T4 has 15GB
VRAM and this pushes it close to the edge. This notebook's defaults
(`--lowram`, disk-cached latents, 768px training resolution) are
chosen to fit on a free T4; if you still hit an out-of-memory error,
see the troubleshooting note in the training cell.

**Colab free-tier limits apply generally** — sessions disconnect
after inactivity and there's a rolling GPU-time cap. LoRA training
(unlike a single image generation) can take 20-60+ minutes depending
on dataset size and epoch count, so budget for that.

## 1. Confirm the GPU is attached

In [ ]:
!nvidia-smi

## 2. Install kohya-ss/sd-scripts

Uses a tarball download instead of `git clone` — GitHub sometimes
rate-limits anonymous `git clone` from shared Colab IPs (you may have
hit this already with the ComfyUI notebook); downloading a tarball
over plain HTTPS sidesteps that entirely.

In [ ]:
%cd /content
!wget -q https://github.com/kohya-ss/sd-scripts/archive/refs/heads/main.tar.gz
!tar xzf main.tar.gz && mv sd-scripts-main sd-scripts && rm main.tar.gz
%cd /content/sd-scripts
!pip install torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124 -q
!pip install --upgrade -r requirements.txt -q
!pip install accelerate -q

You'll likely see a red `ERROR: pip's dependency resolver...` block
listing version conflicts with tensorflow/gradio/numba/etc. **This is
expected and harmless** — those are unrelated packages Colab
pre-installs; sd-scripts doesn't use them. Ignore it and continue.

## 3. Mount Google Drive

Used for reading your training image set and saving the finished LoRA
somewhere that survives the session ending.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 4. Get the base checkpoint — and confirm which architecture it is

**This step matters more than it looks.** You must train against the
*exact same checkpoint* (same model, same architecture family) you
used to generate your training images — a mismatch either crashes
outright (SDXL trainer vs an SD1.5 file, or vice versa) or produces a
LoRA that doesn't actually match what you'll load it with later in
ComfyUI.

Two architecture families need two different trainer scripts later:
- **SDXL** (e.g. Pony Diffusion V6 XL, Illustrious, AniVerse Pony XL) —
  checkpoints are typically 6-7GB. Uses `sdxl_train_network.py`.
- **SD 1.5** (e.g. AniVerse's non-XL "Pruned" releases) — checkpoints
  are typically 2-4GB. Uses `train_network.py`.

**If you're not sure which you generated with**, check: did your
positive prompts use `score_9, score_8_up, score_7_up`? Those tags
only mean anything to Pony-based **SDXL** models — an SD1.5 model
just ignores them as inert text. Generating at 1024-ish resolution
(e.g. 832x1216) is also an SDXL tell; SD1.5 is native at 512.

In [ ]:
MODEL_TYPE = "sdxl"  # @param ["sdxl", "sd15"]
CHECKPOINT_URL = "https://civitai.com/api/download/models/REPLACE_ME"  # @param {type:"string"}
CIVITAI_TOKEN = ""  # @param {type:"string"}

import os
os.makedirs("/content/checkpoints", exist_ok=True)

url = CHECKPOINT_URL
if CIVITAI_TOKEN:
    sep = "&" if "?" in url else "?"
    url = f"{url}{sep}token={CIVITAI_TOKEN}"

!wget --content-disposition "$url" -P /content/checkpoints
!ls -lh /content/checkpoints

Sanity-check the `ls` output against `MODEL_TYPE` above: an `sdxl`
selection should show a ~6-7GB file; `sd15` should show ~2-4GB. A
mismatch here (e.g. you picked `sdxl` but got a 2GB file) means you
grabbed the wrong model version on Civitai — go back to the model's
page and pick the other listed version.

**Already have the checkpoint in Drive instead** (e.g. from the
ComfyUI notebook's 3b step)? Skip the cell above and just set
`CHECKPOINT_PATH` directly to that Drive path in the next cell instead
of the `/content/checkpoints/...` default.

In [ ]:
CHECKPOINT_PATH = "/content/checkpoints/REPLACE_WITH_EXACT_FILENAME_FROM_LS_ABOVE.safetensors"  # @param {type:"string"}

assert os.path.exists(CHECKPOINT_PATH), f"Checkpoint not found at {CHECKPOINT_PATH} — copy the exact filename from the ls output above."
size_gb = os.path.getsize(CHECKPOINT_PATH) / (1024**3)
print(f"Checkpoint found: {CHECKPOINT_PATH} ({size_gb:.1f} GB)")
if MODEL_TYPE == "sdxl" and size_gb < 5:
    print("WARNING: MODEL_TYPE is 'sdxl' but this file is under 5GB — likely an SD1.5 checkpoint. Double check before training.")
if MODEL_TYPE == "sd15" and size_gb > 5:
    print("WARNING: MODEL_TYPE is 'sd15' but this file is over 5GB — likely an SDXL checkpoint. Double check before training.")

## 5. Set up the training image set

1. In Drive, create a folder for this character's training images,
   e.g. `gatefall-lora-training/<character>/`.
2. Put your 15-20 training images directly in that folder.
3. For **each** image, add a matching `.txt` caption file with the
   same base filename (e.g. `faelen_01.png` needs `faelen_01.txt`).
   Caption format: a short **trigger word** unique to this character
   (something the base model won't already associate with anything —
   e.g. `flnwarden`), followed by tags for what's *different* in that
   specific image (pose, expression, crop, background) — leave out
   the constant identity traits (hair color, armor, species markers)
   that are true in every image; the LoRA learns those from the
   trigger word plus the images themselves, not from restating them
   per file. Example `faelen_03.txt`:
   ```
   flnwarden, portrait, upper body, happy expression, slight smile
   ```
4. Update `TRAIN_DATA_DIR` below to match. If you already have the
   images + captions committed in the repo (as Faelen's are, under
   `docs/art-direction/`), pull them straight from GitHub instead of
   uploading by hand — see the optional cell below.

### 5a. Optional — pull a character's images+captions straight from the repo into Drive

Skips manually downloading a GitHub folder and re-uploading it — pulls
the tarball straight into this session and copies just the matching
files into your Drive training folder.

In [ ]:
CHARACTER_PREFIX = "faelen_"  # @param {type:"string"}
REPO_TARBALL_URL = "https://github.com/MohammedEmad333/Gatefall/archive/refs/heads/main.tar.gz"  # @param {type:"string"}
DRIVE_TRAINING_DIR = "/content/drive/MyDrive/gatefall-lora-training/faelen"  # @param {type:"string"}

import os, shutil, tarfile

os.makedirs("/content/repo_tmp", exist_ok=True)
%cd /content/repo_tmp
!wget -q "$REPO_TARBALL_URL" -O repo.tar.gz
with tarfile.open("repo.tar.gz") as tf:
    tf.extractall(".")

extracted_dirs = [d for d in os.listdir(".") if os.path.isdir(d)]
repo_root = extracted_dirs[0]
src = os.path.join(repo_root, "docs", "art-direction")

os.makedirs(DRIVE_TRAINING_DIR, exist_ok=True)
copied = 0
for fname in os.listdir(src):
    if fname.startswith(CHARACTER_PREFIX) and (fname.endswith(".png") or fname.endswith(".txt")):
        shutil.copy(os.path.join(src, fname), DRIVE_TRAINING_DIR)
        copied += 1

print(f"Copied {copied} files to {DRIVE_TRAINING_DIR}")

In [ ]:
TRAIN_DATA_DIR = "/content/drive/MyDrive/gatefall-lora-training/faelen"  # @param {type:"string"}
TRIGGER_WORD = "flnwarden"  # @param {type:"string"}

import os
images = [f for f in os.listdir(TRAIN_DATA_DIR) if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))]
captions = [f for f in os.listdir(TRAIN_DATA_DIR) if f.lower().endswith(".txt")]
print(f"Found {len(images)} images and {len(captions)} caption files in {TRAIN_DATA_DIR}")
missing = [f for f in images if os.path.splitext(f)[0] + ".txt" not in captions]
if missing:
    print("WARNING — these images have no matching .txt caption file:")
    for m in missing:
        print(" ", m)
else:
    print("Every image has a matching caption file. Good to proceed.")

## 6. Write the dataset config

Resolution defaults differ by architecture: SDXL trains natively
around 1024px but that's tight on a free T4's VRAM (see the OOM note
in step 7) — **768 is the safer default for a free T4**; drop to 640
if you still hit out-of-memory. SD1.5 trains natively around 512-768.

`shuffle_caption` is also set by architecture: **off for SDXL, on for
SD1.5**. The SDXL trainer caches text-encoder outputs once (step 7),
and kohya refuses to combine that with caption shuffling — leaving it
on aborts the run seconds after `Loading dataset config from`, before
the checkpoint loads. The cell handles this from `MODEL_TYPE`
automatically; no need to change anything.

In [ ]:
NUM_REPEATS = 10  # @param {type:"integer"}
TRAIN_RESOLUTION = 768  # @param {type:"integer"}

min_bucket = 320 if MODEL_TYPE == "sd15" else 512
max_bucket = 1024 if MODEL_TYPE == "sd15" else 1536

# shuffle_caption randomises caption token order every step so the LoRA doesn't
# over-fit to caption word order. But the SDXL trainer (step 7) passes
# --cache_text_encoder_outputs, which encodes each caption exactly once and
# reuses the cached result — shuffling would have nothing to act on, so kohya
# rejects the combination with an assertion ("...shuffle_caption... cannot be
# used") that aborts the run seconds after "Loading dataset config from",
# before the checkpoint even loads. So enable it only for SD1.5, which encodes
# captions live every step.
shuffle_caption = "false" if MODEL_TYPE == "sdxl" else "true"

dataset_toml = f"""
[general]
caption_extension = '.txt'
shuffle_caption = {shuffle_caption}

[[datasets]]
resolution = {TRAIN_RESOLUTION}
batch_size = 1
enable_bucket = true
min_bucket_reso = {min_bucket}
max_bucket_reso = {max_bucket}

  [[datasets.subsets]]
  image_dir = '{TRAIN_DATA_DIR}'
  num_repeats = {NUM_REPEATS}
"""

with open("/content/dataset_config.toml", "w") as f:
    f.write(dataset_toml)

print(dataset_toml)

## 7. Train

Branches the trainer script and flags by `MODEL_TYPE` from step 4.

**Memory settings baked in for a free T4:**
- `--lowram` — loads the checkpoint straight to VRAM instead of
  staging it in system RAM first. Without this, loading a 6-7GB SDXL
  checkpoint can get the whole process **killed by the OS (SIGKILL)**
  for exceeding Colab's ~12-13GB system RAM ceiling — a completely
  different failure from a CUDA OOM, and easy to mistake for one.
- `--cache_latents --cache_latents_to_disk` — encodes training images
  once up front and caches to disk instead of holding them in memory,
  reducing both RAM and VRAM pressure (and speeds up a re-run).
- `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` — reduces CUDA
  memory fragmentation, per PyTorch's own suggestion when a
  `torch.OutOfMemoryError` is thrown.

**If it still throws `torch.OutOfMemoryError: CUDA out of memory`**
(a real VRAM overflow, distinct from the RAM-based SIGKILL above):
1. First, lower `TRAIN_RESOLUTION` in step 6 to `640` and re-run step
   6, then this cell.
2. Still failing? Lower `--network_dim`/`--network_alpha` below (try
   `16`/`8`) — a smaller LoRA network uses less memory, at some cost
   to how much detail it can capture.
3. As a last resort, `Runtime -> Change runtime type` and check if a
   High-RAM option is available on your account — it doesn't add
   VRAM, but rules out any remaining system-RAM contribution to the
   crash.

**The `steps: ...` progress bar often won't appear in this cell's
output at all, even though training is genuinely running.** kohya
updates that line in place using carriage returns, and Colab's output
panel frequently fails to render those — the cell can look frozen on
`Loading dataset config from` for the entire run. Don't interrupt
based on that alone. Confirm it's actually training via the
**Terminal** panel (bottom-left):
- `nvidia-smi` — `GPU-Util` sitting near `100%` means it's computing,
  not stuck. `0%` for more than a couple of minutes past dataset
  loading means something's actually wrong (see the OOM/SIGKILL notes
  above, or check the images aren't on a slow mount).
- `ls -la /content/lora_output` — with `--save_every_n_epochs=2`
  below, a new numbered `.safetensors` checkpoint should appear every
  couple of epochs. Growing files here are the clearest sign of
  genuine progress, and each one is already a usable (if partially
  trained) LoRA if you need to bail early.</cell id="cell-18">


In [ ]:
OUTPUT_NAME = "faelen_lora"  # @param {type:"string"}
OUTPUT_DIR = "/content/lora_output"  # @param {type:"string"}
MAX_TRAIN_EPOCHS = 10  # @param {type:"integer"}
NETWORK_DIM = 32  # @param {type:"integer"}
NETWORK_ALPHA = 16  # @param {type:"integer"}

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.makedirs(OUTPUT_DIR, exist_ok=True)

%cd /content/sd-scripts

if MODEL_TYPE == "sdxl":
    train_script = "sdxl_train_network.py"
    extra_flags = "--network_train_unet_only --no_half_vae --cache_text_encoder_outputs"
else:
    train_script = "train_network.py"
    extra_flags = "--clip_skip=2"

cmd = f'''accelerate launch --num_cpu_threads_per_process 1 {train_script} \
  --pretrained_model_name_or_path="{CHECKPOINT_PATH}" \
  --dataset_config="/content/dataset_config.toml" \
  --output_dir="{OUTPUT_DIR}" \
  --output_name="{OUTPUT_NAME}" \
  --save_model_as=safetensors \
  --network_module=networks.lora \
  --network_dim={NETWORK_DIM} \
  --network_alpha={NETWORK_ALPHA} \
  {extra_flags} \
  --learning_rate=1e-4 \
  --optimizer_type="AdamW8bit" \
  --lr_scheduler="cosine" \
  --max_train_epochs={MAX_TRAIN_EPOCHS} \
  --save_every_n_epochs=2 \
  --mixed_precision="fp16" \
  --gradient_checkpointing \
  --cache_latents \
  --cache_latents_to_disk \
  --lowram \
  --sdpa'''

print(cmd)
!{cmd}

Success looks like a repeating progress line such as
`steps: 5%|▌ | 10/190 [00:32<08:30, 2.8s/it, loss=0.0812]` — if you
see that ticking upward, it's training; let it run to completion
(20-60+ min depending on dataset size and epoch count).

## 8. Save the trained LoRA to Drive (so it survives the session)

In [ ]:
import shutil, glob, os

DRIVE_LORA_DIR = "/content/drive/MyDrive/gatefall-loras"  # @param {type:"string"}
os.makedirs(DRIVE_LORA_DIR, exist_ok=True)

for f in glob.glob(os.path.join(OUTPUT_DIR, "*.safetensors")):
    shutil.copy(f, DRIVE_LORA_DIR)
    print("Saved:", os.path.join(DRIVE_LORA_DIR, os.path.basename(f)))

## 9. Use it in ComfyUI

Use `gatefall_lora_inference.ipynb` — it launches ComfyUI on Colab
with your checkpoint and this trained LoRA already wired to load, and
covers the `LoraLoader` node setup, strength, and trigger-word usage
in detail. **Load the same base checkpoint you trained against** — a
LoRA trained on AniVerse Pony XL, for example, won't work correctly
loaded on top of Pony Diffusion V6 XL, even though both are
Pony-based.